# Weekly Demand Forecasting for Store or Group Datasets
Edit the CSV filename and date, demand, and optional group columns in the first code cell. The pipeline rolls daily or transaction-level sales into Monday-start weekly totals, validates each group, compares a random forest with seasonal-naive and recent-average strategies using up to four chronological windows, exports the selected per-group strategies for `app.py`, and saves forecasts as CSV. Group labels may be text or numeric; the API exposes stable integer IDs and original labels.
Forecasting requires at least 65 weeks per group: 52 weeks to build lag features and at least 13 supervised training examples. Horizon-matched backtesting is available when there are at least 65 plus the forecast-horizon number of weekly observations. Each demand value must be nonnegative and numeric. Set `DATE_FORMAT` explicitly for ambiguous dates. Transactions in a week are summed.
Only past demand and calendar features are used; planned holidays, promotions, weather, and other future external values are not modeled.


In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from forecasting import FEATURE_COLUMNS, MIN_HISTORY, build_feature_frame, evaluate_forecast_strategies, recursive_forecast

# Edit these settings to match your CSV.
BASE_DIR = Path.cwd()
DATA_FILE = 'Walmart.csv'  # CSV file inside data/; keep local datasets out of Git.
DATE_COLUMN = 'Date'
DEMAND_COLUMN = 'Weekly_Sales'
GROUP_COLUMN = 'Store'  # Set to None for one overall series.
DATE_FORMAT = '%d-%m-%Y'  # Set to None to infer; specify it for ambiguous dates.
DATE_DAYFIRST = False
FORECAST_HORIZON = 4
BUNDLE_PATH = BASE_DIR / 'artifacts' / 'demand_forecasting_bundle.joblib'
FORECAST_PATH = BASE_DIR / 'artifacts' / 'next_4_weeks_forecast.csv'
MODEL_PARAMETERS = {'n_estimators': 100, 'min_samples_leaf': 2, 'random_state': 42, 'n_jobs': -1}
GROUP_ID, GROUP_VALUE = '_forecast_store_id', '_forecast_group_value'
OBSERVATION_DATE, DEMAND_VALUE = '_forecast_date', '_forecast_demand'


def load_weekly_dataset(path):
    if not path.is_file():
        raise FileNotFoundError(f'Dataset not found: {path}')
    frame = pd.read_csv(path)
    required = [DATE_COLUMN, DEMAND_COLUMN]
    if GROUP_COLUMN is not None:
        required.append(GROUP_COLUMN)
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f'Dataset is missing configured columns {missing}. Available columns: {frame.columns.tolist()}')
    if DATE_COLUMN == DEMAND_COLUMN or (GROUP_COLUMN is not None and GROUP_COLUMN in (DATE_COLUMN, DEMAND_COLUMN)):
        raise ValueError('Date, demand, and group columns must be distinct.')
    parsed_dates = (
        pd.to_datetime(frame[DATE_COLUMN], format=DATE_FORMAT, errors='coerce')
        if DATE_FORMAT is not None
        else pd.to_datetime(frame[DATE_COLUMN], dayfirst=DATE_DAYFIRST, errors='coerce')
    )
    demand_values = pd.to_numeric(frame[DEMAND_COLUMN], errors='coerce')
    if parsed_dates.isna().any():
        raise ValueError('Date column has missing or invalid dates; check DATE_FORMAT.')
    if demand_values.isna().any():
        raise ValueError('Demand column has missing or non-numeric values.')
    if not np.isfinite(demand_values.to_numpy(dtype=float)).all():
        raise ValueError('Demand values must all be finite.')
    if (demand_values < 0).any():
        raise ValueError('Demand values cannot be negative.')
    if GROUP_COLUMN is None:
        group_values = pd.Series(['All data'] * len(frame), index=frame.index)
    else:
        group_values = frame[GROUP_COLUMN].astype('string').str.strip()
        if group_values.isna().any() or group_values.eq('').any():
            raise ValueError('Group column contains missing or blank values.')
    labels = sorted(group_values.unique().tolist())
    group_ids = {label: index for index, label in enumerate(labels, start=1)}
    data = pd.DataFrame(index=frame.index)
    data[GROUP_ID] = group_values.map(group_ids).astype('int64')
    data[GROUP_VALUE] = group_values.astype(str)
    data[OBSERVATION_DATE] = parsed_dates - pd.to_timedelta(parsed_dates.dt.weekday, unit='D')
    data[DEMAND_VALUE] = demand_values
    data = data.groupby([GROUP_ID, GROUP_VALUE, OBSERVATION_DATE], as_index=False)[DEMAND_VALUE].sum()
    data = data.sort_values([GROUP_ID, OBSERVATION_DATE]).reset_index(drop=True)
    minimum_rows = MIN_HISTORY + 13 + FORECAST_HORIZON
    for _, group in data.groupby(GROUP_ID, sort=True):
        label = group[GROUP_VALUE].iloc[0]
        gaps = group[OBSERVATION_DATE].diff().dropna()
        if not gaps.eq(pd.Timedelta(days=7)).all():
            raise ValueError(f'Group {label!r} has missing or irregular weeks; observations must be seven days apart.')
        if len(group) < minimum_rows:
            raise ValueError(f'Group {label!r} has {len(group)} weeks; at least {minimum_rows} are required for horizon-matched validation with 13 training examples.')
    return data


def build_group_features(data):
    feature_sets = {}
    for group_id, group in data.groupby(GROUP_ID, sort=True):
        group = group.sort_values(OBSERVATION_DATE).reset_index(drop=True)
        demand = group[DEMAND_VALUE].astype(float)
        dates = group[OBSERVATION_DATE]
        features = build_feature_frame(demand, dates)
        features['forecast_date'] = dates
        features['target'] = demand
        feature_sets[int(group_id)] = features.dropna(subset=FEATURE_COLUMNS + ['target']).reset_index(drop=True)
    return feature_sets


def create_model():
    return RandomForestRegressor(
        n_estimators=MODEL_PARAMETERS['n_estimators'],
        min_samples_leaf=MODEL_PARAMETERS['min_samples_leaf'],
        random_state=MODEL_PARAMETERS['random_state'],
        n_jobs=MODEL_PARAMETERS['n_jobs'],
    )


def forecast_group(group_id, group_value, model, strategy, history, latest_date, horizon):
    predictions = recursive_forecast(model, history, pd.Timestamp(latest_date), horizon, strategy)
    return [
        {'Group_ID': group_id, 'Group_Value': group_value, 'Forecast_Date': (pd.Timestamp(latest_date) + pd.Timedelta(weeks=step)).strftime('%Y-%m-%d'), 'Predicted_Demand': round(prediction, 2)}
        for step, prediction in enumerate(predictions, start=1)
    ]

DATA_PATH = (BASE_DIR / 'data' / DATA_FILE).resolve()
data = load_weekly_dataset(DATA_PATH)
feature_sets = build_group_features(data)
print(f'Loaded {len(data):,} weekly rows across {data[GROUP_ID].nunique()} group(s).')
print(f'Configured columns: date={DATE_COLUMN!r}, demand={DEMAND_COLUMN!r}, group={GROUP_COLUMN!r}')
print(f'Date range: {data[OBSERVATION_DATE].min():%Y-%m-%d} to {data[OBSERVATION_DATE].max():%Y-%m-%d}')


In [ ]:
# Evaluate several demand-only strategies on chronological rolling windows.
validation_rows = []
selected_strategies = {}
for group_id, features in feature_sets.items():
    group_history = data.loc[data[GROUP_ID].eq(group_id)].sort_values(OBSERVATION_DATE)
    evaluation = evaluate_forecast_strategies(
        group_history[DEMAND_VALUE].astype(float).tolist(),
        group_history[OBSERVATION_DATE].reset_index(drop=True),
        FORECAST_HORIZON,
        create_model,
    )
    selected_strategies[group_id] = str(evaluation['selected_model'])
    validation_rows.append({
        'Group_ID': group_id,
        'Group_Value': group_history[GROUP_VALUE].iloc[0],
        'Selected_Model': evaluation['selected_model'],
        'Validation_Windows': evaluation['validation_windows'],
        'MAE': evaluation['mae'],
        'RMSE': evaluation['rmse'],
        'WAPE_percent': evaluation['wape_percent'],
    })
validation_metrics = pd.DataFrame(validation_rows)
print(f'Per-group rolling-origin validation (up to four {FORECAST_HORIZON}-week windows):')
print(validation_metrics[['MAE', 'RMSE', 'WAPE_percent']].median().round(2).to_string())

# Refit on all observations and save the API-compatible model bundle.
models, histories, metadata = {}, {}, {}
for group_id, features in feature_sets.items():
    strategy = selected_strategies[group_id]
    models[group_id] = create_model().fit(features[FEATURE_COLUMNS], features['target']) if strategy == 'random_forest' else None
    group_history = data.loc[data[GROUP_ID].eq(group_id)].sort_values(OBSERVATION_DATE)
    histories[group_id] = group_history[DEMAND_VALUE].astype(float).tolist()
    metadata[group_id] = {'latest_date': group_history[OBSERVATION_DATE].iloc[-1].strftime('%Y-%m-%d'), 'training_rows': len(features), 'group_value': str(group_history[GROUP_VALUE].iloc[0])}
bundle = {
    'models': models, 'histories': histories, 'metadata': metadata,
    'strategies': selected_strategies,
    'feature_columns': FEATURE_COLUMNS, 'target_column': DEMAND_COLUMN,
    'date_column': DATE_COLUMN, 'group_column': GROUP_COLUMN,
    'model_type': 'Rolling-origin-selected demand strategy', 'model_parameters': MODEL_PARAMETERS,
    'sklearn_version': __import__('sklearn').__version__,
    'created_at': pd.Timestamp.now(tz='UTC').isoformat(),
}
BUNDLE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, BUNDLE_PATH)
print()
print(f'Saved API model bundle for {len(models)} group(s): {BUNDLE_PATH}')

forecast_rows = []
for group_id in sorted(models):
    forecast_rows.extend(forecast_group(group_id, metadata[group_id]['group_value'], models[group_id], selected_strategies[group_id], histories[group_id], metadata[group_id]['latest_date'], FORECAST_HORIZON))
forecast_frame = pd.DataFrame(forecast_rows)
FORECAST_PATH.parent.mkdir(parents=True, exist_ok=True)
forecast_frame.to_csv(FORECAST_PATH, index=False)
print(f'Forecast rows: {len(forecast_frame):,}; groups: {forecast_frame.Group_ID.nunique()}; weeks/group: {FORECAST_HORIZON}')
print(f'Forecast CSV saved to {FORECAST_PATH}')
print(forecast_frame.head(12).to_string(index=False))
